# Week 14: Prefix Tries, KMP & Rabin-Karp String Matching

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

O(L) prefix autocomplete, linear-time KMP failure function, and rolling-hash search.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-14/labs/W14_String_Matching_and_Tries.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Prefix Trie & Autocomplete

Inserting and querying strings of length $L$ in a Trie takes $O(L)$ time independent of dictionary size $N$.

In [1]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False


class PrefixTrie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        cur = self.root
        for ch in word:
            cur = cur.children.setdefault(ch, TrieNode())
        cur.is_end = True

    def autocomplete(self, prefix):
        cur = self.root
        for ch in prefix:
            if ch not in cur.children:
                return []
            cur = cur.children[ch]
        out = []

        def dfs(node, buf):
            if node.is_end:
                out.append(prefix + "".join(buf))
            for ch in sorted(node.children):
                buf.append(ch); dfs(node.children[ch], buf); buf.pop()

        dfs(cur, [])
        return out


t = PrefixTrie()
for crop in ["clove", "cloves", "coconut", "cashew", "cassava", "cardamom"]:
    t.insert(crop)
print("Autocomplete 'ca':", t.autocomplete("ca"))
assert t.autocomplete("ca") == ["cardamom", "cashew", "cassava"]


Autocomplete 'ca': ['cardamom', 'cashew', 'cassava']


## 2. Knuth-Morris-Pratt (KMP) & Rabin-Karp Rolling Hash

KMP preprocesses the pattern LPS array in $O(m)$ so the text pointer never backs up ($O(n + m)$ worst case); Rabin-Karp updates a polynomial rolling hash in $O(1)$ per slide.

In [2]:
def compute_lps(pat):
    lps = [0] * len(pat)
    length, i = 0, 1
    while i < len(pat):
        if pat[i] == pat[length]:
            length += 1; lps[i] = length; i += 1
        elif length > 0:
            length = lps[length - 1]
        else:
            i += 1
    return lps


def kmp_search(text, pat):
    lps = compute_lps(pat)
    hits, i, j = [], 0, 0
    while i < len(text):
        if text[i] == pat[j]:
            i += 1; j += 1
            if j == len(pat):
                hits.append(i - j); j = lps[j - 1]
        elif j > 0:
            j = lps[j - 1]
        else:
            i += 1
    return hits


def rabin_karp(text, pat, base=256, mod=1_000_000_007):
    n, m = len(text), len(pat)
    hp = pow(base, m - 1, mod)
    ph = wh = 0
    for i in range(m):
        ph = (ph * base + ord(pat[i])) % mod
        wh = (wh * base + ord(text[i])) % mod
    hits = []
    for i in range(n - m + 1):
        if ph == wh and text[i : i + m] == pat:
            hits.append(i)
        if i < n - m:
            wh = ((wh - ord(text[i]) * hp) * base + ord(text[i + m])) % mod
    return hits


text, pattern = "pemba_clove_pemba_cassava_pemba", "pemba"
assert kmp_search(text, pattern) == rabin_karp(text, pattern) == [0, 12, 26]
print("Matches for 'pemba':", kmp_search(text, pattern))


Matches for 'pemba': [0, 12, 26]
